# 1.3 Update MLflow Experiments

**Objective:** Update experiment metadata using tags.

This notebook is fully independent. It does not depend on helper folders, custom utility modules, or files outside this notebook folder.

In [1]:

import os
import sys
import json
import time
import math
import pickle
import tempfile
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import mlflow
import mlflow.sklearn
import mlflow.pyfunc
from mlflow.tracking import MlflowClient
from mlflow.models.signature import infer_signature

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report,
    mean_absolute_error, mean_squared_error, r2_score
)
from sklearn.datasets import (
    load_iris, load_diabetes, load_digits, load_linnerud,
    load_wine, load_breast_cancer, make_classification, make_regression
)
from sklearn.linear_model import LogisticRegression, LinearRegression, Ridge, ElasticNet
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor, GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.multioutput import MultiOutputRegressor

warnings.filterwarnings("ignore")


def find_project_root(start_path=None):
    """Find the mlflow_for_ml_dev folder from wherever Jupyter is started."""
    start = Path(start_path or Path.cwd()).resolve()
    for parent in [start] + list(start.parents):
        if parent.name == "mlflow_for_ml_dev" and (parent / "notebooks").exists():
            return parent
        if (parent / "datasets").exists() and (parent / "mlruns").exists() and (parent / "notebooks").exists():
            return parent
    # Fallback: if user started Jupyter inside notebooks subfolder
    for parent in [start] + list(start.parents):
        if parent.name == "notebooks":
            return parent.parent
    return start

PROJECT_ROOT = find_project_root()
DATASETS_DIR = PROJECT_ROOT / "datasets"
MLRUNS_DIR = PROJECT_ROOT / "mlruns"
MODELS_DIR = PROJECT_ROOT / "models"
ARTIFACTS_DIR = PROJECT_ROOT / "notebooks" / "artifacts_generated"

for folder in [DATASETS_DIR, MLRUNS_DIR, MODELS_DIR, ARTIFACTS_DIR, MLRUNS_DIR / ".trash"]:
    folder.mkdir(parents=True, exist_ok=True)

# IMPORTANT: Every notebook logs to the same mlruns folder inside this project.
mlflow.set_tracking_uri(MLRUNS_DIR.as_uri())
client = MlflowClient()

print("Project root      :", PROJECT_ROOT)
print("Datasets folder   :", DATASETS_DIR)
print("MLflow runs folder:", MLRUNS_DIR)
print("Models folder     :", MODELS_DIR)
print("Tracking URI      :", mlflow.get_tracking_uri())


Project root      : /Users/giridharsripathi/Documents/training/cgi_mlflow_new
Datasets folder   : /Users/giridharsripathi/Documents/training/cgi_mlflow_new/datasets
MLflow runs folder: /Users/giridharsripathi/Documents/training/cgi_mlflow_new/mlruns
Models folder     : /Users/giridharsripathi/Documents/training/cgi_mlflow_new/models
Tracking URI      : file:///Users/giridharsripathi/Documents/training/cgi_mlflow_new/mlruns


## 2. Create or reuse an experiment

In [2]:
experiment_name = "independent_01_update_experiment"
experiment = client.get_experiment_by_name(experiment_name)
experiment_id = client.create_experiment(experiment_name) if experiment is None else experiment.experiment_id
print("Experiment id:", experiment_id)

Experiment id: 992898500112710361


## 3. Add and update experiment tags

In [3]:
client.set_experiment_tag(experiment_id, "project", "MLflow training")
client.set_experiment_tag(experiment_id, "difficulty", "basic")
client.set_experiment_tag(experiment_id, "difficulty", "beginner")  # update existing tag

exp = client.get_experiment(experiment_id)
print(json.dumps(exp.tags, indent=2))

{
  "difficulty": "beginner",
  "project": "MLflow training"
}
